# MODEL_BASELINE — Exploration historique

> **Statut : notebook exploratoire.** La baseline officielle est évaluée par `scripts/Models/evaluate_final.py` sur les mêmes cas test que les modèles servis.

Baseline canonique : films partageant au moins un genre avec les 1 à 3 seeds, au moins 5 notes dans apprentissage+validation, triés par note moyenne décroissante, avec exclusion des seeds uniquement.

Le protocole officiel est défini dans `scripts/evaluation_protocol.py`. Les anciennes sorties enregistrées dans ce notebook ne sont pas les résultats finaux ; voir `results/final_evaluation.json`.


## 1. Chargement


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_STATE = 42
# Aimé = note strictement au-dessus de la moyenne de l'utilisateur (train)
MAX_SEEDS = 3
TRAIN_FRAC = 0.25
MIN_SUPPORT = 22

np.random.seed(RANDOM_STATE)

GENRES = [
    "Action", "Adventure", "Animation", "Children's", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical",
    "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western",
]

DATA_DIR = next(
    (p for p in [Path("dataset"), Path("../dataset")] if (p / "u.data.csv").exists()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("dataset/ introuvable")

notes = pd.read_csv(DATA_DIR / "u.data.csv")
films = pd.read_csv(DATA_DIR / "u.item.csv")
users = pd.read_csv(DATA_DIR / "u.user.csv", dtype={"zip_code": "string"}).drop(
    columns=["occupation"], errors="ignore"
)
df = notes.merge(users, on="user_id", how="left").merge(
    films, left_on="item_id", right_on="movie_id", how="left"
)
df = df.loc[df["unknown"] != 1].drop(columns=["unknown"]).copy()
df["genres"] = df[GENRES].apply(lambda row: [g for g in GENRES if row[g] == 1], axis=1)
print(f"Notes : {len(df):,} | users : {df['user_id'].nunique()} | films : {df['item_id'].nunique()}")
display(df[["user_id", "item_id", "rating", "age", "gender", "title", "genres"]].head(3))


## 2. Split sans fuite


In [ ]:
train = df.sample(frac=TRAIN_FRAC, random_state=RANDOM_STATE)
test = df.drop(train.index)
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)
print(f"Train {len(train):,} | Test {len(test):,}")

movies = df.drop_duplicates("item_id").set_index("item_id")
movie_genres = movies["genres"].to_dict()
movie_title = movies["title"].to_dict()
film_genre_X = movies[GENRES].astype(float)
all_movie_ids = film_genre_X.index.to_numpy()
print(f"Films catalogue : {len(all_movie_ids)}")


## 3. Stats train + cas d'évaluation


In [ ]:
film_stats = (
    train.groupby("item_id")["rating"]
    .agg(n_ratings="count", mean_rating="mean")
    .reset_index()
)
eligible = film_stats.loc[film_stats["n_ratings"] >= MIN_SUPPORT].set_index("item_id")
mean_rating = eligible["mean_rating"]
print(f"Films eligible (>= {MIN_SUPPORT} notes) : {len(eligible)}")

user_mean = train.groupby("user_id")["rating"].mean()
train = train.copy()
test = test.copy()
train["user_mean"] = train["user_id"].map(user_mean)
test["user_mean"] = test["user_id"].map(user_mean)
liked_train = train.loc[train["rating"] > train["user_mean"]]
liked_test = test.loc[test["rating"] > test["user_mean"]]
print(
    f"Aimé (note > moyenne user) : train {len(liked_train):,} | test {len(liked_test):,} "
    f"| moyenne des moyennes user = {user_mean.mean():.2f}"
)
train_counts = train.groupby("item_id").size()
cand_A = liked_train.copy()
cand_A["pop"] = cand_A["item_id"].map(train_counts)


def _get_top_seeds(df_u, max_n=MAX_SEEDS):
    sorted_u = df_u.sort_values(["rating", "pop"], ascending=[False, False])
    return list(sorted_u["item_id"].unique()[:max_n])


seeds_per_user = (
    cand_A.groupby("user_id", group_keys=False).apply(_get_top_seeds).rename("query_ids")
)
user_demo = train.drop_duplicates("user_id").set_index("user_id")[["age", "gender"]]
relevant = liked_test.groupby("user_id")["item_id"].apply(set).rename("relevant_ids")
liked_lists = liked_train.groupby("user_id")["item_id"].apply(set)

eval_cases = pd.concat([seeds_per_user, user_demo, relevant], axis=1, join="inner").reset_index()
eval_cases["relevant_ids"] = eval_cases.apply(
    lambda r: r["relevant_ids"] - set(r["query_ids"]), axis=1
)
eval_cases = eval_cases.loc[eval_cases["relevant_ids"].map(len) > 0].reset_index(drop=True)
eval_cases["liked_train_ids"] = eval_cases["user_id"].map(liked_lists)
print(f"Cas d'évaluation (1-{MAX_SEEDS} films) : {len(eval_cases)}")
display(eval_cases.head(3))


## 4. Recommenders dummy & baseline


In [ ]:
def _seen(query_ids, liked_train_ids):
    if isinstance(query_ids, (int, np.integer)):
        seeds = {int(query_ids)}
    else:
        seeds = set(int(x) for x in query_ids)
    s = set(liked_train_ids) if liked_train_ids is not None else set()
    return s.union(seeds), seeds


def recommend_dummy(query_ids, k=10, age=None, gender=None, liked_train_ids=None, user_id=None):
    seen, seeds = _seen(query_ids, liked_train_ids)
    pool = np.array([m for m in all_movie_ids if m not in seen])
    seed_val = sum(seeds)
    local = np.random.default_rng(RANDOM_STATE + seed_val)
    if len(pool) == 0:
        return []
    return local.choice(pool, size=min(k, len(pool)), replace=False).tolist()


def recommend_baseline(query_ids, k=10, age=None, gender=None, liked_train_ids=None, user_id=None):
    seen, seeds = _seen(query_ids, liked_train_ids)
    union_g = set()
    for qid in seeds:
        union_g.update(movie_genres.get(qid, []))
    if not union_g:
        return []
    scores = [
        (mid, float(m))
        for mid, m in mean_rating.items()
        if mid not in seen and union_g & set(movie_genres.get(mid, []))
    ]
    scores.sort(key=lambda x: (-x[1], x[0]))
    return [m for m, _ in scores[:k]]


ex = eval_cases.iloc[0]
qids = ex["query_ids"]
kw = dict(age=ex["age"], gender=ex["gender"], liked_train_ids=None, user_id=None)
print("Seeds :", [movie_title.get(i) for i in qids])
print("Dummy    :", [movie_title.get(i) for i in recommend_dummy(qids, 5, **kw)])
print("Baseline :", [movie_title.get(i) for i in recommend_baseline(qids, 5, **kw)])


## 5. Évaluation mode produit


In [ ]:
import sys
from pathlib import Path

_ROOT = next(p for p in [Path("."), Path("..")] if (p / "scripts" / "Models" / "eval_diagnostics.py").exists())
sys.path.insert(0, str((_ROOT / "scripts" / "Models").resolve()))
from eval_diagnostics import collect_method_eval, save_eval_bundle

methods = [
    ("dummy_random", recommend_dummy),
    ("baseline_overlap_mean", recommend_baseline),
]
method_data = {}
for name, fn in methods:
    method_data[name] = collect_method_eval(fn, eval_cases)

results_product = pd.DataFrame([
    {
        "method": name,
        "P@5": float(np.mean(d["P@5"])),
        "P@10": float(np.mean(d["P@10"])),
    }
    for name, d in method_data.items()
])
print(f"Situation produit — {len(eval_cases)} cas (seed={RANDOM_STATE})")
display(results_product.round(4))

out = save_eval_bundle(
    "baseline",
    method_data,
    film_stats.set_index("item_id")["n_ratings"],
    n_cases=len(eval_cases),
    random_state=RANDOM_STATE,
    train_frac=TRAIN_FRAC,
)
print(f"Export diagnostics → {out}")


## Lecture

La baseline doit clairement battre le dummy. C'est le plancher à dépasser pour les modèles CF (`MODEL_CF`) et cluster (`MODEL_CLUSTER`).

Pour la **variance** des P@5 et le **biais de popularité** (comparaison transverse) : section 3 de `CONCLUSION.ipynb` après les trois notebooks modèles.
